In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_5024_Alipur_Delhi_DPCC_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,157.01,224.01,2.85,25.22,15.74,23.28,11.56,1.03,17.87,...,NaN,11.80,72.37,NaN,209.27,0.00,0.00,33.66,973.78,NaN
1,2024-01-02,149.00,208.58,3.93,24.24,15.89,19.63,11.26,1.05,20.92,...,NaN,11.81,70.45,NaN,200.06,0.00,0.00,62.88,974.11,NaN
2,2024-01-03,169.25,235.07,15.33,24.54,25.47,22.85,6.13,1.42,21.03,...,1.72,9.15,80.58,NaN,95.06,0.00,0.00,30.14,974.24,NaN
3,2024-01-04,175.42,248.66,7.98,29.59,22.23,26.42,5.70,1.23,5.14,...,0.68,9.24,82.70,NaN,265.53,0.00,0.00,18.29,974.02,NaN
4,2024-01-05,118.19,171.18,2.55,21.21,13.36,38.92,5.49,1.06,13.68,...,0.49,10.27,83.41,NaN,226.78,0.00,0.00,13.95,974.49,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,133.25,171.04,3.67,33.55,20.83,34.12,6.83,0.95,26.05,...,0.62,13.48,90.46,NaN,143.83,1.14,1.14,8.45,997.02,NaN
362,2024-12-28,78.23,104.94,7.11,47.58,31.02,39.70,6.07,0.88,8.38,...,0.69,13.74,91.00,NaN,186.73,0.10,0.06,7.24,996.16,NaN
363,2024-12-29,82.33,120.95,2.95,24.58,15.47,25.97,5.99,0.76,31.87,...,0.30,13.41,90.31,NaN,259.78,0.00,0.00,74.22,997.32,NaN
364,2024-12-30,91.71,117.53,2.63,22.35,14.04,26.61,5.35,0.82,42.79,...,0.19,12.07,90.28,NaN,256.43,0.00,0.00,68.46,996.52,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 23)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)', 'O Xylene (µg/m³)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp              0
PM2.5 (µg/m³)          0
PM10 (µg/m³)           0
NO (µg/m³)             0
NO2 (µg/m³)            0
NOx (ppb)              0
NH3 (µg/m³)            0
SO2 (µg/m³)            0
CO (mg/m³)             0
Ozone (µg/m³)          0
Benzene (µg/m³)        0
Toluene (µg/m³)        0
Eth-Benzene (µg/m³)    0
MP-Xylene (µg/m³)      0
AT (°C)                0
RH (%)                 0
WD (deg)               0
RF (mm)                0
TOT-RF (mm)            0
SR (W/mt2)             0
BP (mmHg)              0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 21)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         157.01        224.01        2.85        25.22   
1  2024-01-02         149.00        208.58        3.93        24.24   
2  2024-01-03         169.25        235.07       15.33        24.54   
3  2024-01-04         175.42        248.66        7.98        29.59   
4  2024-01-05         118.19        171.18        2.55        21.21   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  ...  \
0      15.74        23.28        11.56        1.03          17.87  ...   
1      15.89        19.63        11.26        1.05          20.92  ...   
2      25.47        22.85         6.13        1.42          21.03  ...   
3      22.23        26.42         5.70        1.23           5.14  ...   
4      13.36        38.92         5.49        1.06          13.68  ...   

   Toluene (µg/m³)  Eth-Benzene (µg/m³)  MP-Xylene (µg/m³)  AT (°C)  RH (%)  \
0             4.92        

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,Toluene (µg/m³),Eth-Benzene (µg/m³),MP-Xylene (µg/m³),AT (°C),RH (%),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg)
0,2024-01-01,1.174085,0.350499,-0.449496,-0.365843,-0.420249,-0.409241,0.279944,0.306320,-1.207160,...,-0.253531,-0.179652,-0.142782,-1.529908,0.711572,0.362269,0.0,0.0,-2.000017,-1.119939
1,2024-01-02,1.039842,0.195473,-0.136751,-0.443771,-0.403655,-0.845408,0.221299,0.368796,-1.073897,...,-0.200833,-0.179652,-0.142782,-1.528796,0.584063,0.232410,0.0,0.0,-1.432025,-1.071857
2,2024-01-03,1.379221,0.461618,3.164451,-0.419916,0.656150,-0.460625,-0.781540,1.524600,-1.069091,...,0.150487,2.568795,-0.142782,-1.824746,1.256805,-1.248065,0.0,0.0,-2.068440,-1.052915
3,2024-01-04,1.482627,0.598157,1.036044,-0.018346,0.297719,-0.034018,-0.865599,0.931079,-1.763368,...,1.056891,0.700163,0.476324,-1.814733,1.397596,1.155521,0.0,0.0,-2.298786,-1.084970
4,2024-01-05,0.523482,-0.180285,-0.536370,-0.684713,-0.683541,1.459704,-0.906651,0.400034,-1.390232,...,0.220751,0.357580,-0.058358,-1.700135,1.444747,0.609155,0.0,0.0,-2.383149,-1.016489
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,0.775880,-0.181691,-0.212042,0.296549,0.142842,0.886115,-0.644701,0.056416,-0.849754,...,-0.095437,0.715735,0.307477,-1.342992,1.912943,-0.560420,0.0,0.0,-2.490061,2.266236
362,2024-12-28,-0.146226,-0.845798,0.784111,1.412197,1.270130,1.552913,-0.793269,-0.162250,-1.621803,...,0.120625,0.871454,0.504465,-1.314064,1.948805,0.044460,0.0,0.0,-2.513581,2.140931
363,2024-12-29,-0.077513,-0.684946,-0.420539,-0.416735,-0.450118,-0.087792,-0.808908,-0.537105,-0.595462,...,0.190888,-0.218581,-0.593040,-1.350780,1.902982,1.074447,0.0,0.0,-1.211592,2.309948
364,2024-12-30,0.079691,-0.719307,-0.513204,-0.594062,-0.608315,-0.011313,-0.934019,-0.349677,-0.118338,...,-0.104220,-0.685740,-0.902593,-1.499868,1.900989,1.027213,0.0,0.0,-1.323558,2.193384


In [10]:
df.to_excel('Alipur2024.xlsx', index=False)